In [ ]:
from email.message import EmailMessage
import os
import smtplib
import requests
from openai.types.responses import ResponseTextDeltaEvent
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, SQLiteSession, OpenAIChatCompletionsModel, set_tracing_disabled, RunHooks
from IPython.display import Markdown, display
import asyncio
from agents.extensions.visualization import draw_graph
from agents import ModelSettings, model_settings
from dotenv import load_dotenv
load_dotenv(override=True)
set_tracing_disabled(disabled=True)

ollama_client = AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
local_llm = OpenAIChatCompletionsModel(model="llama3.2", openai_client=ollama_client)
temp_settings = ModelSettings(temperature=0.0)
require_tool = ModelSettings(tool_choice="required")

# Define custom local logging hooks
class LocalOrchestrationLogger(RunHooks):
    async def on_agent_start(self, context, agent):
        print(f"\n🔄 [AGENT START] Active Agent switched to: {agent.name}")

    async def on_handoff(self, context, from_agent, to_agent):
        print(f"🔀 [HANDOFF] Control passing from '{from_agent.name}' ➡️ '{to_agent.name}'")

    async def on_tool_start(self, context, tool_definition, arguments):
        print(f"🛠️ [TOOL CALL] Running tool: {tool_definition.name} with args: {arguments}")

    async def on_llm_end(self, context, agent, response):
        print(f"🤖 [LLM RESPONSE] {agent.name} generated a response.")

# Run your multi-agent conversation loop
# res = await runner.run(agent=triage_agent, input="I have a billing issue", hooks=LocalOrchestrationLogger())

smtp_server = os.getenv("EMAIL_SMTP_SERVER")
mail_app_password = os.getenv("EMAIL_APP_PASSWORD")
email_address = os.getenv("EMAIL_ADDRESS")
USE_EMAIL = True

def send_email(subject, text_body, html_body):
    msg = EmailMessage()
    msg["From"] = email_address
    msg["To"] = email_address
    msg["Subject"] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype="html")

    with smtplib.SMTP(smtp_server, 587) as server:
        server.starttls()
        server.login(email_address, mail_app_password)
        server.send_message(msg)

@function_tool
def send_message(subject, text_body, html_body):
    """ Send the message either to the email or print based on the USE_EMAIL variable """
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        print(f"Subject: {subject}\n\n{text_body}")

@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_email(subject, text_body, html_body)
    return "Email sent successfully"


In [21]:
intro= """
You are a sales agent working for ComplAI,
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write short emails.
Do not use ASCII codes in the email body or html
"""

professional_email_instructions = intro + "Your email style is professional, serious, with gravities and credibility."
humorous_email_instruction = intro + "Your email is witty, engaging and humorous."
executive_email_instructions = intro + "Your email is concise, to the point, in the style of a busy senior executive."

professional_agent = Agent(name="Professional Sales Agent", model=local_llm, instructions=professional_email_instructions)
humorous_agent = Agent(name="Humorous Sales Agent", model=local_llm, instructions=humorous_email_instruction)
executive_agent = Agent(name="Executive Sales Agent", model=local_llm, instructions=executive_email_instructions)

In [ ]:
from agents import ModelSettings, model_settings


decision = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Then use your tool to send the email.
"""

sales_sender = Agent(name="Sales Sender", instructions=decision, model=local_llm, tools=[send_email_tool], model_settings=ollama_settings)

In [23]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = professional_agent.as_tool(tool_name="sales_email_writer_1", tool_description=description)
tool2 = humorous_agent.as_tool(tool_name="sales_email_writer_2", tool_description=description)
tool3 = executive_agent.as_tool(tool_name="sales_email_writer_3", tool_description=description)

tools = [tool1, tool2, tool3, send_email_tool]

In [24]:
instructions = """ 
You are a Sales Manager at ComplAI, Your goal is to find the single best cold sales email using the sales_writer tools and then send the email to the users.
"""

task = """ 
Follow these steps one by one in order and each step is mandatory:

1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.

2. Evaluate and Select: Review the drafts and choose the single best email using your judgement of which one is the most effective that you will most likely reply to if you were a user. 
Do not alter the selected email. Use the same subject and body as drafted. 

3. Use your tool send_email_tool to send the best email (and only the best email) to the user. Only send 1 email. 
You must use the same subject, text_body and html_body as in the selected best draft and pass the parameters to the send_email_tool.
Make sure the tool sends the email and Print 'Email sent successfully' only after the tool sends the email.
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=local_llm, model_settings=ollama_settings)

In [26]:

result = await Runner.run(sales_manager, task, hooks=LocalOrchestrationLogger())
print(result.final_output)


🔄 [AGENT START] Active Agent switched to: Sales Manager
🤖 [LLM RESPONSE] Sales Manager generated a response.
🛠️ [TOOL CALL] Running tool: Sales Manager with args: FunctionTool(name='sales_email_writer_1', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x000002A396975FA0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)
🛠️ [TOOL CALL] Running tool: Sales Manager with args: FunctionTool(name='sales_email_writer_2', description='Use 